In [1]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.7 MB/s  0:00:00m0:00:010m

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [7]:
# ============================================================
# graphrag_gat_pipeline.py
# Подготовка графа GraphRAG и обучение GAT (link prediction)
# ============================================================
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.utils import coalesce
from sentence_transformers import SentenceTransformer

ARTIFACTS_DIR = Path("./graphrag_preprocessed_rubert/output")
entities = pd.read_parquet(ARTIFACTS_DIR / "entities.parquet")
relationships = pd.read_parquet(ARTIFACTS_DIR / "relationships.parquet")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def make_node_text(row):
    title = str(row.get("title", ""))
    desc = str(row["description"]) if pd.notna(row.get("description")) else ""
    return f"{title}. {desc}"[:512]

node_texts = entities.apply(make_node_text, axis=1).tolist()
embeddings = model.encode(
    node_texts, batch_size=64, show_progress_bar=True,
    convert_to_tensor=True, normalize_embeddings=True,
)

title_to_idx = {}
for idx, title in enumerate(entities["title"].astype(str)):
    if title not in title_to_idx:
        title_to_idx[title] = idx

src = relationships["source"].astype(str).map(title_to_idx)
tgt = relationships["target"].astype(str).map(title_to_idx)
mask = src.notna() & tgt.notna()
src = src[mask].astype(int).values
tgt = tgt[mask].astype(int).values

edge_index = torch.tensor([src, tgt], dtype=torch.long)
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
edge_index = coalesce(edge_index, num_nodes=len(entities))

data = Data(x=embeddings, edge_index=edge_index, num_nodes=len(entities))
print(data)

Batches: 100%|██████████| 3/3 [00:00<00:00, 18.89it/s]

Data(x=[149, 384], edge_index=[2, 260], num_nodes=149)


Подотовка данных для обучения:
- data.edge_index - матрица смежности, кто с кем сосед
- data.x -эмбеддинги текстов title+description + длина

In [ ]:
# code